In [4]:
MODEL_PATH = "runs/detect/train18/weights/best.pt"

In [5]:
import torch
# import torchvision.models as models 
from pytorch_nndct.apis import torch_quantizer

In [ ]:
# model = torch.load(MODEL_PATH)
# model.eval()  # 記得如果是推論，要設 eval 模式

from ultralytics import YOLO

# 載入 YOLO 模型
yolo_model = YOLO(MODEL_PATH)  # 或者你自己的.pt

# 取出原生 PyTorch 模型
torch_model = yolo_model.model

print(type(torch_model))
# 模型的通道數和輸入尺寸
print("Channels:", torch_model.yaml['nc'])  # 類別數量
# <class 'torch.nn.modules.module.Module'>

<class 'ultralytics.nn.tasks.DetectionModel'>
Channels: 4
Recommended input size: tensor(1024.)


In [ ]:
input_shape = (1, 3, 640, 640) 
dummy_input = torch.randn(input_shape)

# 3. 初始化 Vitis AI 量化器 (模式設為 'calib' 進行校準)
quantizer = torch_quantizer('calib', torch_model, (dummy_input), output_dir='quantize_result')
quant_model = quantizer.quant_model

In [ ]:
import torch
# 假設您使用的是影像分類模型，請替換成您自己的模型定義
import torchvision.models as models 
from pytorch_nndct.apis import torch_quantizer

def main():
    # 1. 初始化並載入您的模型與 .pt 權重
    model = models.resnet18() # 請換成您的模型架構
    model.load_state_dict(torch.load('your_model.pt', map_location='cpu'))
    model.eval()

    # 2. 定義模型的輸入張量形狀 (Batch Size, Channels, Height, Width)
    input_shape = (1, 3, 224, 224) 
    dummy_input = torch.randn(input_shape)

    # 3. 初始化 Vitis AI 量化器 (模式設為 'calib' 進行校準)
    quantizer = torch_quantizer('calib', model, (dummy_input), output_dir='quantize_result')
    quant_model = quantizer.quant_model

    # 4. 進行校準 (Calibration)
    # 【重要】這裡必須使用約 100~1000 張真實的資料讓模型推論過，以收集數值分佈。
    # 以下僅為示範，實務上請使用您的 DataLoader 跑迴圈
    print("開始校準...")
    with torch.no_grad():
        # for images, _ in calibration_loader:
        #     quant_model(images)
        quant_model(dummy_input) # 僅示範用，請換成真實數據迴圈

    # 5. 匯出量化設定檔與中繼的 xmodel
    print("匯出量化模型...")
    quantizer.export_quant_config()
    quantizer.export_xmodel(deploy_check=False, output_dir='quantize_result')

if __name__ == '__main__':
    main()